# State Data

## Processing the employment data

Using state_M2024_dl from the OEWS to get the employment per state at the occupation level

Filter on:
- AREA_TYPE: [2,3] 
- O_GROUP: Use `detailed`

Features to keep
- PRIM_STATE: First two letters of the state (common codification)
- AREA: State code in FIPS
- AREA_TITLE: Name of the state
- OCC_CODE: SOC code, !! Not the same as ONET!!
- TOT_EMP: Number of employees for the occ

In [17]:
import pandas as pd
import numpy as np

# Load
OEWS_state_df = pd.read_csv("./data/OEWS/state_M2024_dl.csv")
onet_mapping = pd.read_csv("./data/onet_code_to_id_mapping.csv")

OEWS_state_df = pd.merge(OEWS_state_df,onet_mapping, left_on = "OCC_CODE", right_on = "onet_code", how="left")
OEWS_state_df.drop(columns=["onet_code", "OCC_CODE"], inplace=True)

# Filter
OEWS_state_df = OEWS_state_df[
    OEWS_state_df["AREA_TYPE"].isin([2, 3]) &
    (OEWS_state_df["O_GROUP"] == "detailed")
].copy()

# Keep relevant features
OEWS_state_df = OEWS_state_df[
    ["AREA", "onet_id", "TOT_EMP"]
].copy()

# Optional cleanup
OEWS_state_df["AREA"] = OEWS_state_df["AREA"].astype(str).str.zfill(2)
OEWS_state_df["onet_id"] = OEWS_state_df["onet_id"].astype(int)
OEWS_state_df["TOT_EMP"] = pd.to_numeric(OEWS_state_df["TOT_EMP"], errors="coerce")

OEWS_state_df.replace({"TOT_EMP": {np.nan: 0}}, inplace=True)

OEWS_state_df.rename(columns={
    "AREA": "state_code",
    "onet_id": "onet_code",
    "TOT_EMP": "weight"
}, inplace=True)

OEWS_state_df["weight"] = OEWS_state_df["weight"].astype(int)
OEWS_state_df["ai_exposure"] = 0
OEWS_state_df["ai_penetration"] = 1
OEWS_state_df.reset_index(drop=False, inplace=True)
OEWS_state_df.rename(columns={"index": "id"}, inplace=True)
print(OEWS_state_df.head())
print(OEWS_state_df.shape)


OEWS_state_df.to_csv("./data/OEWS/processed_state_OEWS.csv", index = False)

   id state_code  onet_code  weight  ai_exposure  ai_penetration
0   2         01          0     830            0               1
1   3         01          1       0            0               1
2   4         01          2       0            0               1
3   5         01          3      50            0               1
4   6         01          4       0            0               1
(36367, 6)


## Generating the GeoJson at the state level

In [14]:
from pathlib import Path
import geopandas as gpd

# Paths
data_dir = Path("./data/maps")
state_zip = data_dir / "cb_2024_us_state_500k.zip"
out_geojson = data_dir / "us_states.geojson"

if not state_zip.exists():
    raise FileNotFoundError(
        f"Missing {state_zip}. Please place the Census state shapefile zip there."
    )

# Read local Census state boundaries
states_gdf = gpd.read_file(f"zip://{state_zip.resolve()}")

# Build 2-digit state FIPS key for joins with OEWS AREA
states_gdf["state_code"] = states_gdf["STATEFP"].astype(str).str.zfill(2)

# Keep only fields needed by frontend/join
states_out = states_gdf[["state_code", "NAME", "geometry"]].copy()

# Export GeoJSON
states_out.to_file(out_geojson, driver="GeoJSON")

print(f"Saved {out_geojson}")
print("Rows:", len(states_out))
print("Columns:", list(states_out.columns))
print(states_out[["state_code", "NAME"]].head())


Saved data/maps/us_states.geojson
Rows: 56
Columns: ['state_code', 'NAME', 'geometry']
  state_code          NAME
0         35    New Mexico
1         46  South Dakota
2         06    California
3         21      Kentucky
4         01       Alabama
